In [1]:
# ==============================================================================
# CELL 1: SYSTEM INFRASTRUCTURE & DYNAMIC WORKLOAD ALLOCATION
# ==============================================================================
import os
import sys
import logging
import time
import shutil
import subprocess
import pandas as pd
import numpy as np
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from typing import List

# --- 1. SYSTEM LOGGING CONFIGURATION ---
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler("ablation_extraction.log")
    ])
logger = logging.getLogger("DistPipeline")

# --- 2. DEPENDENCY RESOLUTION & MODEL ACQUISITION ---
def install_package(command: str):
    try:
        subprocess.run(command.split(), check=True, stdout=subprocess.DEVNULL)
    except subprocess.CalledProcessError:
        logger.critical(f"FATAL: Subprocess execution failed: {command}")
        sys.exit(1)

logger.info("Initializing runtime environment for distributed latent space ablation...")
if shutil.which('openslide-show-properties') is None:
    install_package("apt-get install -y openslide-tools")
install_package("pip install openslide-python -q")  # Removed timm dependency

import openslide

# Acquire the frozen, compiled TorchScript model to bypass library versioning conflicts
if not os.path.exists("torchscript_model.pt"):
    logger.info("Downloading compiled TorchScript CTransPath architecture...")
    subprocess.run("wget -q https://huggingface.co/kaczmarj/CTransPath/resolve/main/torchscript_model.pt", shell=True)
    logger.info("TorchScript model acquired successfully.")

# --- 3. DISTRIBUTED WORKLOAD CONFIGURATION ---
class Config:
    # MASTER CONTROL: Change this ID (1 through 7) for each Kaggle Session
    WORKER_ID: int = 7
    TOTAL_WORKERS: int = 7
    
    INPUT_ROOT = Path("/kaggle/input")
    OUTPUT_DIR = Path(f"/kaggle/working/features_node_{WORKER_ID}")
    
    BATCH_SIZE: int = 128
    STRIDE: int = 224
    TISSUE_THRESHOLD: int = 220
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    MIN_DISK_GB: float = 4.0 

Config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
logger.info(f"Compute Node {Config.WORKER_ID}/{Config.TOTAL_WORKERS} initialized.")

# --- 4. DYNAMIC ROUND-ROBIN ALLOCATOR ---
def get_target_slides() -> List[str]:
    logger.info("Scanning mounted directories for WSI artifacts...")
    
    # Dynamically locate all files across all mounted directories
    normal_slides = sorted([str(p) for p in Config.INPUT_ROOT.glob("**/normal_*.tif")])
    tumor_slides = sorted([str(p) for p in Config.INPUT_ROOT.glob("**/tumor_*.tif")])
    
    master_list = normal_slides + tumor_slides
    total_slides = len(master_list)
    
    if total_slides == 0:
        logger.critical("Zero .tif artifacts detected. Verify Kaggle dataset mounts.")
        sys.exit(1)
        
    logger.info(f"Global registry compiled: {len(normal_slides)} Normal | {len(tumor_slides)} Tumor | Total: {total_slides}")
    
    # Mathematical round-robin distribution to ensure equal load and class representation
    worker_index = Config.WORKER_ID - 1
    assigned_slides = master_list[worker_index::Config.TOTAL_WORKERS]
    
    return assigned_slides

MY_SLIDES = get_target_slides()
logger.info(f"Allocation complete: Node {Config.WORKER_ID} assigned {len(MY_SLIDES)} total slides.")

2026-03-25 19:48:24 | INFO | Initializing runtime environment for distributed latent space ablation...
2026-03-25 19:48:38 | INFO | Downloading compiled TorchScript CTransPath architecture...
2026-03-25 19:48:38 | INFO | TorchScript model acquired successfully.
2026-03-25 19:48:38 | INFO | Compute Node 7/7 initialized.
2026-03-25 19:48:38 | INFO | Scanning mounted directories for WSI artifacts...
2026-03-25 19:48:39 | INFO | Global registry compiled: 159 Normal | 111 Tumor | Total: 270
2026-03-25 19:48:39 | INFO | Allocation complete: Node 7 assigned 38 total slides.


In [2]:
# ==============================================================================
# CELL 2: FOUNDATION MODEL INITIALIZATION (JIT) & TISSUE MAPPING
# ==============================================================================
class WSI_Patch_Dataset(Dataset):
    def __init__(self, slide_path: str, stride: int = 224):
        self.slide = openslide.OpenSlide(str(slide_path))
        self.stride = stride
        self.norm = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        self._generate_coords()

    def _generate_coords(self):
        # Extract thumbnail and dynamically read exact dimensions to prevent aspect-ratio drift
        thumb = self.slide.get_thumbnail((512, 512)).convert("L")
        thumb_w, thumb_h = thumb.size 
        w, h = self.slide.dimensions
        
        # Binary thresholding for tissue isolation
        mask = np.array(thumb) < Config.TISSUE_THRESHOLD
        ys, xs = np.where(mask)
        
        # Precise coordinate projection back to Level 0 resolution
        self.coords = [
            (int(x * (w / thumb_w)), int(y * (h / thumb_h))) 
            for x, y in zip(xs, ys)
        ]

    def __len__(self) -> int:
        return len(self.coords)

    def __getitem__(self, idx: int) -> torch.Tensor:
        x, y = self.coords[idx]
        patch = self.slide.read_region((x, y), 0, (224, 224)).convert("RGB")
        t = transforms.ToTensor()(patch)
        return self.norm(t)

logger.info("Initializing weights for CTransPath foundation model (TorchScript version)...")
# Loading the JIT compiled model bypasses all library versioning conflicts
feature_extractor = torch.jit.load("torchscript_model.pt")
feature_extractor = feature_extractor.to(Config.DEVICE).eval()
logger.info("Model loaded into VRAM. Output embedding dimensionality configured to 768.")

2026-03-25 19:48:39 | INFO | Initializing weights for CTransPath foundation model (TorchScript version)...
2026-03-25 19:48:39 | INFO | Model loaded into VRAM. Output embedding dimensionality configured to 768.


In [3]:
# ==============================================================================
# CELL 3: LATENT SPACE PROJECTION & TENSOR SERIALIZATION
# ==============================================================================
failed_slides = []
logger.info("Executing sequential latent space projection pipeline...")

for i, slide_path in enumerate(MY_SLIDES):
    # Resource Monitor: Hardware safeguard against I/O corruption during extraction.
    total, used, free = shutil.disk_usage("/")
    if (free // (2**30)) < Config.MIN_DISK_GB:
        logger.critical(f"CRITICAL WARNING: Storage volume approaching capacity threshold (< {Config.MIN_DISK_GB} GB). Terminating loop.")
        break

    slide_name = Path(slide_path).stem
    save_path = Config.OUTPUT_DIR / f"{slide_name}.pt"
    
    if save_path.exists():
        logger.info(f"Artifact exists: Bypassing inference for {slide_name}.")
        continue
        
    try:
        dataset = WSI_Patch_Dataset(slide_path, stride=Config.STRIDE)
        
        # Dimensionality Enforcement: Maintain tensor integrity for slides lacking valid tissue masks.
        if len(dataset) == 0:
            torch.save(torch.zeros(1, 768), save_path)
            logger.warning(f"Tissue threshold zero-match for {slide_name}. Serialized null-tensor.")
            continue
            
        # I/O Optimization: Configuring DataLoader with pinned memory and prefetching for maximum GPU throughput.
        loader = DataLoader(
            dataset, batch_size=Config.BATCH_SIZE, num_workers=2, 
            pin_memory=True, prefetch_factor=2, shuffle=False
        )
        features_list = []
        
        with torch.no_grad():
            for batch in loader:
                batch = batch.to(Config.DEVICE)
                feats = feature_extractor(batch)
                features_list.append(feats.cpu())
        
        if features_list:
            full_feats = torch.cat(features_list, dim=0)
            torch.save(full_feats, save_path)
            logger.info(f"Progress: [{i+1}/{len(MY_SLIDES)}] | {slide_name} processed. Tensor shape: {full_feats.shape}.")
            
    except Exception as e:
        logger.error(f"Inference failed for {slide_name}. Exception: {str(e)}")
        failed_slides.append({'slide': slide_name, 'error': str(e)})

zip_name = f"ctranspath_node{Config.WORKER_ID}_data"
logger.info(f"Compressing serialized artifacts into archive: {zip_name}.zip")
shutil.make_archive(zip_name, 'zip', Config.OUTPUT_DIR)

if failed_slides:
    pd.DataFrame(failed_slides).to_csv(Config.OUTPUT_DIR / "failures.csv", index=False)
    logger.warning(f"Execution terminated. Logged {len(failed_slides)} structural or processing anomalies.")
else:
    logger.info(f"Execution phase complete for Node {Config.WORKER_ID}. Serialized tensors archived successfully.")

2026-03-25 19:48:39 | INFO | Executing sequential latent space projection pipeline...
2026-03-25 19:49:09 | INFO | Progress: [1/38] | normal_007 processed. Tensor shape: torch.Size([6562, 768]).
2026-03-25 19:49:31 | INFO | Progress: [2/38] | normal_014 processed. Tensor shape: torch.Size([5044, 768]).
2026-03-25 19:49:44 | INFO | Progress: [3/38] | normal_021 processed. Tensor shape: torch.Size([2790, 768]).
2026-03-25 19:50:15 | INFO | Progress: [4/38] | normal_028 processed. Tensor shape: torch.Size([7691, 768]).
2026-03-25 19:50:34 | INFO | Progress: [5/38] | normal_035 processed. Tensor shape: torch.Size([4311, 768]).
2026-03-25 19:50:42 | INFO | Progress: [6/38] | normal_042 processed. Tensor shape: torch.Size([1862, 768]).
2026-03-25 19:51:09 | INFO | Progress: [7/38] | normal_049 processed. Tensor shape: torch.Size([6313, 768]).
2026-03-25 19:51:49 | INFO | Progress: [8/38] | normal_056 processed. Tensor shape: torch.Size([8407, 768]).
2026-03-25 19:52:35 | INFO | Progress: [9/